In [1]:
USE_GPU = True

if USE_GPU:
    import cuml.accel
    cuml.accel.install()

    #import cudf.pandas
    #cudf.pandas.install()

import pandas as pd
import seaborn as sns
import numpy as np
import pathlib
import optuna
import json
from joblib import dump, load

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC


sns.set_theme(context="notebook", style="white", palette="pastel")

# path handling
base_dir = pathlib.Path("/kaggle/input/playground-series-s6e2/")
train_path = base_dir / "train.csv"
test_path = base_dir / "test.csv"
sample_path = base_dir / "sample_submission.csv"
submission_file = "submission.csv"
submission_path = pathlib.Path("/kaggle/working/submission.csv")
hyperparams_path = pathlib.Path("/kaggle/working/data/hyperparams/")
hyperparams_path.mkdir(parents=True, exist_ok=True)
oof_path = pathlib.Path("/kaggle/working/data/oof/")
oof_path.mkdir(parents=True, exist_ok=True)
preds_path = pathlib.Path("/kaggle/working/data/preds/")
preds_path.mkdir(parents=True, exist_ok=True)
models_path = pathlib.Path("/kaggle/working/data/models/")
models_path.mkdir(parents=True, exist_ok=True)

# csv file handling
train = pd.read_csv(train_path, index_col="id")
test = pd.read_csv(test_path, index_col="id")
sample = pd.read_csv(sample_path, index_col="id")

# column handling
LABEL_COLUMN = "Heart Disease"
NUMERICAL_FEATURE_COLUMNS = ["Age", "BP", "Cholesterol", "Max HR", "ST depression", "Slope of ST", "Number of vessels fluro"]
CATEGORICAL_FEATURE_COLUMNS = ["Sex", "Chest pain type", "EKG results", "Exercise angina", "FBS over 120", "Thallium"]
assert set([LABEL_COLUMN] + NUMERICAL_FEATURE_COLUMNS + CATEGORICAL_FEATURE_COLUMNS) == set(list(train.columns))
assert len([LABEL_COLUMN]) + len(NUMERICAL_FEATURE_COLUMNS) + len(CATEGORICAL_FEATURE_COLUMNS) == len(train.columns)

# global parameters
MASTER_SEED = 123
N_CV_SPLITS = 7
N_TRIALS_XGB = 128
N_TRIALS_LOGR = 256
N_TRIALS_KNN = 5

In [2]:
def preprocess(df_in):
    df = df_in.copy()
    df = _add_feature_max_hr_deviation(df)
    df = _add_feature_ischemia(df)
    df = _add_feature_st_severity(df)
    return df

def _add_feature_max_hr_deviation(df):
    df["Theoretical Max HR"] = 220 - df["Age"]
    df["Max HR deviation"] = df["Theoretical Max HR"] - df["Max HR"]
    df = df.drop(columns=["Theoretical Max HR"])

    global NUMERICAL_FEATURE_COLUMNS
    if "Max HR deviation" not in NUMERICAL_FEATURE_COLUMNS:
        NUMERICAL_FEATURE_COLUMNS += ["Max HR deviation"]
    return df
    
def _add_feature_ischemia(df_in):
    df = df_in.copy()
    ischemia_score = (
        (df["Exercise angina"]==1).astype(int)
        + (df["ST depression"]>0).astype(int)
        + (df["Thallium"].isin([6,7])).astype(int)
        + (df["Slope of ST"]==3).astype(int)
    )
    df["Ischemia score"] = ischemia_score

    global NUMERICAL_FEATURE_COLUMNS
    if "Ischemia score" not in NUMERICAL_FEATURE_COLUMNS:
        NUMERICAL_FEATURE_COLUMNS += ["Ischemia score"]
    return df

def _add_feature_st_severity(df_in):
    df = df_in.copy()
    st_severity = df["ST depression"] * df["Slope of ST"]
    df["ST severity"] = st_severity

    global NUMERICAL_FEATURE_COLUMNS
    if "ST severity" not in NUMERICAL_FEATURE_COLUMNS:
        NUMERICAL_FEATURE_COLUMNS += ["ST severity"]
    return df

train = preprocess(train)
test = preprocess(test)

In [3]:
# define preprocessing strategy
def make_pipeline(estimator):
    numeric_pipe = Pipeline(steps=[
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline(steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, NUMERICAL_FEATURE_COLUMNS),
            ("cat", categorical_pipe, CATEGORICAL_FEATURE_COLUMNS),
        ],
        verbose_feature_names_out=True,
    ).set_output(transform="pandas")
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("estimator", estimator)
    ])
    return pipe

In [4]:
label_encoder = LabelEncoder()

X = train[NUMERICAL_FEATURE_COLUMNS + CATEGORICAL_FEATURE_COLUMNS]
y = pd.Series(label_encoder.fit_transform(train[LABEL_COLUMN]), index=X.index)

X_pred = test[NUMERICAL_FEATURE_COLUMNS + CATEGORICAL_FEATURE_COLUMNS]

In [5]:
CV = StratifiedKFold(n_splits=N_CV_SPLITS, shuffle=True, random_state=MASTER_SEED)

# HYPERPARAMETER TUNING XGB

In [7]:
XGB_BASE = dict(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_estimators=300,
        n_jobs=1,
        random_state=MASTER_SEED,
        device="cuda" if USE_GPU else "cpu",
    )

def xgb_objective(trial: optuna.Trial) -> float:
    params = {
        # integer
        "max_depth": trial.suggest_int("max_depth", 2, 10),

        # continuous (log where appropriate)
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 3e-1, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-2, 1e2, log=True),
        "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 1e2, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 0.2, 10.0, log=True),
    }

    pipeline = make_pipeline(estimator=XGBClassifier(**XGB_BASE, **params))

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=CV,
        scoring="roc_auc",
        n_jobs=1,
        error_score="raise",
    )
    return float(np.mean(scores))

xgb_study = optuna.create_study(direction="maximize")
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS_XGB, show_progress_bar=True)

print("Best xgb score (CV):", xgb_study.best_value)
print("Best xgb params:", xgb_study.best_params)

best_xgb_pipeline = make_pipeline(estimator=XGBClassifier(**XGB_BASE, **xgb_study.best_params))
best_xgb_pipeline.fit(X, y)

with open(hyperparams_path / "xgb.json", "w", encoding="utf-8") as f:
    json.dump(best_xgb_pipeline["estimator"].get_params(), f, indent=2, ensure_ascii=False)

In [ ]:
"""
Best xgb score (CV): 0.9553375185118799
Best xgb params: {'max_depth': 4, 'learning_rate': 0.1390119111444948, 'subsample': 0.9421851413326763, 'colsample_bytree': 0.5242140754796143, 'min_child_weight': 0.0592402425740781, 'gamma': 0.0025294220979719863, 'reg_alpha': 9.295103193400207e-06, 'reg_lambda': 0.0029219717414297906, 'scale_pos_weight': 0.9393912081476044}
"""

# HYPERPARAMETER TUNING LOGR

In [6]:
LOGR_BASE = dict(
    solver="saga",
    max_iter=5000,
    n_jobs=1,
    random_state=MASTER_SEED,
)

def lr_objective(trial: optuna.Trial) -> float:
    penalty = trial.suggest_categorical("penalty", ["l2", "l1", "elasticnet"])

    params = {
        "penalty": penalty,
        "C": trial.suggest_float("C", 1e-4, 1e4, log=True),
        "tol": trial.suggest_float("tol", 1e-6, 1e-2, log=True),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
    }

    # only valid for elasticnet with saga
    if penalty == "elasticnet":
        params["l1_ratio"] = trial.suggest_float("l1_ratio", 0.0, 1.0)

    pipeline = make_pipeline(estimator=LogisticRegression(**LOGR_BASE, **params))

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=CV,
        scoring="roc_auc",
        n_jobs=1,
        error_score="raise",
    )
    return float(np.mean(scores))


logr_study = optuna.create_study(direction="maximize")
logr_study.optimize(lr_objective, n_trials=N_TRIALS_LOGR, show_progress_bar=True)

print("Best lr score (CV):", logr_study.best_value)
print("Best lr params:", logr_study.best_params)

best_logr_pipeline = make_pipeline(
    estimator=LogisticRegression(**LOGR_BASE, **logr_study.best_params)
)
best_logr_pipeline.fit(X, y)

with open(hyperparams_path / "logr.json", "w", encoding="utf-8") as f:
    json.dump(best_logr_pipeline["estimator"].get_params(), f, indent=2, ensure_ascii=False)

In [ ]:
"""
Best lr score (CV): 0.9526051820626268
Best lr params: {'penalty': 'l2', 'C': 0.07906077518091878, 'tol': 4.227668863153959e-05, 'fit_intercept': False, 'class_weight': 'balanced'}
"""

# HYPERPARAMETER TUNING KNN

In [7]:
KNN_BASE = dict()

def knn_objective(trial: optuna.Trial) -> float:
    params = {
        "n_neighbors": trial.suggest_int("n_neighbors", 1, 49),
    }

    pipeline = make_pipeline(estimator=KNeighborsClassifier(**KNN_BASE, **params))

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=CV,
        scoring="roc_auc",
        n_jobs=1,
        error_score="raise",
    )
    return float(np.mean(scores))

knn_study = optuna.create_study(direction="maximize")
knn_study.optimize(knn_objective, n_trials=N_TRIALS_KNN, show_progress_bar=True)

print("Best knn score (CV):", knn_study.best_value)
print("Best knn params:", knn_study.best_params)

best_knn_pipeline = make_pipeline(
    estimator=KNeighborsClassifier(**KNN_BASE, **knn_study.best_params)
)
best_knn_pipeline.fit(X, y)

with open(hyperparams_path / "knn.json", "w", encoding="utf-8") as f:
    json.dump(best_knn_pipeline["estimator"].get_params(), f, indent=2, ensure_ascii=False)

In [ ]:
"""
Best knn score (CV): 0.947590672477407
Best knn params: {'n_neighbors': 32}
"""

# GENERATE OOF AND BASE PREDS

In [8]:
def train_across_folds(name, model_constructor):
    # load best hyperparams
    with open(hyperparams_path / f"{name}.json", "r", encoding="utf-8") as f:
        hyperparams = json.load(f)

    # init oof and inference for each of the seeds
    oof = np.zeros(len(train), dtype=np.float32)
    preds_by_fold = np.zeros((N_CV_SPLITS, len(test)), dtype=np.float32)

    # get oof and inference for each fold
    for cv_idx, (tr_idx, va_idx) in enumerate(CV.split(X, y)):
        pipeline = make_pipeline(estimator=model_constructor(**hyperparams))
        pipeline.fit(X.iloc[tr_idx], y[tr_idx])
        dump(pipeline, models_path / f"{name}-pipeline-cv{cv_idx}.joblib")
        oof[va_idx] = pipeline.predict_proba(X.iloc[va_idx])[:, 1].astype(np.float32)
        preds_by_fold[cv_idx, :] = pipeline.predict_proba(X_pred)[:, 1].astype(np.float32) 
    preds = preds_by_fold.mean(axis=0)
    np.save(preds_path / f"{name}.npy", preds)
    np.save(oof_path / f"{name}.npy", oof)

train_across_folds("xgb", XGBClassifier)
train_across_folds("logr", LogisticRegression)
train_across_folds("knn", KNeighborsClassifier)

# HYPERPARAMETER TUNING META LEARNER

In [9]:
import json
import optuna
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

base = ["xgb", "logr", "knn"]
X_meta = np.column_stack([np.load(oof_path / f"{n}.npy") for n in base])

META_LOGR_BASE = dict(
    solver="saga",
    max_iter=5000,
    n_jobs=1,
    random_state=MASTER_SEED,
)

def meta_lr_objective(trial: optuna.Trial) -> float:
    penalty = trial.suggest_categorical("penalty", ["l2", "l1", "elasticnet"])

    params = {
        "penalty": penalty,
        "C": trial.suggest_float("C", 1e-4, 1e4, log=True),
        "tol": trial.suggest_float("tol", 1e-6, 1e-2, log=True),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
    }

    if penalty == "elasticnet":
        params["l1_ratio"] = trial.suggest_float("l1_ratio", 0.0, 1.0)

    meta = LogisticRegression(**META_LOGR_BASE, **params)

    scores = cross_val_score(
        meta,
        X_meta,
        y,
        cv=CV,
        scoring="roc_auc",
        n_jobs=1,
        error_score="raise",
    )
    return float(np.mean(scores))

meta_study = optuna.create_study(direction="maximize")
meta_study.optimize(meta_lr_objective, n_trials=N_TRIALS_LOGR, show_progress_bar=True)

print("Best meta lr score (CV):", meta_study.best_value)
print("Best meta lr params:", meta_study.best_params)

best_meta = LogisticRegression(**META_LOGR_BASE, **meta_study.best_params)
best_meta.fit(X_meta, y)

with open(hyperparams_path / "meta_logr.json", "w", encoding="utf-8") as f:
    json.dump(best_meta.get_params(), f, indent=2, ensure_ascii=False)

In [ ]:
"""
Best meta lr score (CV): 0.9553375188684196
Best meta lr params: {'penalty': 'l1', 'C': 0.0001010380310622311, 'tol': 0.00030085324006698486, 'fit_intercept': False, 'class_weight': None}
"""

In [18]:
X_meta_test = np.column_stack([np.load(preds_path / f"{n}.npy") for n in base])
meta_pred = best_meta.predict_proba(X_meta_test)[:, 1]

In [20]:
# run inference
#y_pred = best_xgb_pipeline.predict_proba(X_pred)[:, 1]
#y_pred = preds_by_seed.mean(axis=0).mean(axis=0)
y_pred = meta_pred

# instantiate submission dataframe
submission = pd.DataFrame({
    "Heart Disease":y_pred
})
submission.index = X_pred.index
submission.to_csv(submission_path)

# submit csv file
comp = "playground-series-s6e2"
msg = "optuna 128/256/5 trials 7-fold cv stacking xgb/logr/knn into logr meta" 
!kaggle competitions submit -c {comp} -f {submission_path} -m "{msg}"

In [19]:
!kaggle competitions submissions -c {comp} | head -n 20

fileName        date                        description                                                             status                     publicScore  privateScore  
--------------  --------------------------  ----------------------------------------------------------------------  -------------------------  -----------  ------------  
submission.csv  2026-02-26 21:59:29.630000  autogluon best quality 1hr                                              SubmissionStatus.COMPLETE  0.95359                    
submission.csv  2026-02-26 21:48:42.407000  autogluon best quality 1hr                                              SubmissionStatus.COMPLETE  0.88417                    
submission.csv  2026-02-26 18:11:26.937000  optuna 128/256/5 trials 7-fold cv stacking xgb/logr/knn into logr meta  SubmissionStatus.COMPLETE  0.95349                    
submission.csv  2026-02-26 14:32:46.530000  xgb optuna 2 trials 2-fold cv stacking test                             SubmissionStatus.COMPLETE  0.

CV | LB

0.9551882347384169 | 0.95342

0.9551979091333953 | 0.95336 

0.9555028269442882 | 0.95375 

0.9540939338712681 | 0.95232 

In [6]:
from autogluon.tabular import TabularPredictor

pred = TabularPredictor(
    label=LABEL_COLUMN,
    eval_metric="roc_auc",
    problem_type='binary'
).fit(
    train_data=pd.concat([X, y], axis=1).rename(columns={0:LABEL_COLUMN}),
    presets='best_quality_v150',
    time_limit=1*60*60,
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260226_204751"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Sat Jan 17 11:20:45 UTC 2026
CPU Count:          4
Pytorch Version:    2.9.0+cu126
CUDA Version:       12.6
GPU Memory:         GPU 0: 14.56/14.56 GB | GPU 1: 14.56/14.56 GB
Total GPU Memory:   Free: 29.13 GB, Allocated: 0.00 GB, Total: 29.13 GB
GPU Count:          2
Memory Avail:       29.04 GB / 31.35 GB (92.6%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['best_quality_v150']
Using hyperparameters preset: hyperparameters='zeroshot_2025_12_18_cpu'
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=8, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "/kaggle/working/AutogluonModels/ag-20260226_204751

In [15]:
y_pred = pred.predict_proba(
    data=X_pred
)[1]

# instantiate submission dataframe
submission = pd.DataFrame({
    "Heart Disease":y_pred
})
submission.index = X_pred.index
submission.to_csv(submission_path)

# submit csv file
comp = "playground-series-s6e2"
msg = "autogluon best quality 1hr" 
!kaggle competitions submit -c {comp} -f {submission_path} -m "{msg}"

100%|██████████████████████████████████████| 6.86M/6.86M [00:00<00:00, 16.2MB/s]
Successfully submitted to Predicting Heart Disease